In [ ]:
# ==============================================================================# 🚀 DO NOT MODIFY: Standardized Notebook Setup# ==============================================================================# This cell is designed to work in both Google Colab and local environments.# It ensures that the environment is correctly configured by cloning (or# locating) the project repository and installing the necessary dependencies.## ------------------------------------------------------------------------------##  ⚠️  IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY (NOT ON COLAB):##  This cell will automatically find the repository root and configure your#  environment. Just make sure you have run: pip install -e .[dev]## ------------------------------------------------------------------------------import osimport subprocessimport sysfrom pathlib import Path# --- Configuration ---REPO_URL = "https://github.com/BradSegal/ADH-LLM-Tutorials-2025.git"REPO_DIR = Path("ADH-LLM-Tutorials-2025")  # The name of the directory once cloned# --- End of Configuration ---def find_repo_root(start_path: Path) -> Path | None:    """    Find the repository root by looking for pyproject.toml.    Searches upward from start_path until it finds pyproject.toml or hits root.    Args:        start_path: Directory to start searching from.    Returns:        Path to repository root, or None if not found.    """    current = start_path.resolve()    while current \!= current.parent:  # Stop at filesystem root        if (current / "pyproject.toml").exists():            return current        current = current.parent    return Nonedef detect_active_branch(repo_dir: Path) -> str:    """    Determine the active git branch for pulling updates.    Tries multiple methods to detect the current branch name.    Args:        repo_dir: Path to the git repository.    Returns:        Branch name (defaults to 'master' if detection fails).    """    commands = [        "git symbolic-ref --short HEAD",        "git rev-parse --abbrev-ref HEAD",    ]    for cmd in commands:        result = subprocess.run(            cmd, shell=True, cwd=repo_dir, capture_output=True, text=True        )        if result.returncode == 0:            branch = result.stdout.strip()            if branch and not branch.startswith("origin/"):                return branch    return "master"def run_cmd(cmd: str, *, cwd: Path | None = None) -> None:    """    Run a shell command and raise an error if it fails.    Args:        cmd: The command to run.        cwd: Optional working directory for the command.    Raises:        RuntimeError: If the command returns a non-zero exit code.    """    result = subprocess.run(cmd, shell=True, cwd=cwd)    if result.returncode \!= 0:        raise RuntimeError(f"Command failed with exit code {result.returncode}: {cmd}")# --- Detect environment ---try:    import google.colab  # noqa: F401    IN_COLAB = Trueexcept ImportError:    IN_COLAB = False# --- Main setup logic ---if IN_COLAB:    print("☁️  Running in Google Colab. Setting up the environment...")    # Determine repository path    start_dir = Path.cwd()    if start_dir.name == REPO_DIR.name:        repo_path = start_dir    else:        repo_path = start_dir / REPO_DIR    # Clone or update repository    if not repo_path.exists():        print(f"📥 Cloning repository from {REPO_URL}...")        run_cmd(f"git clone --quiet {REPO_URL} {repo_path}")        print(f"✅ Repository cloned to {repo_path}")    else:        print(f"📂 Repository already exists at {repo_path}")        active_branch = detect_active_branch(repo_path)        print(f"🔄 Pulling latest changes from branch '{active_branch}'...")        run_cmd(f"git pull origin {active_branch} --quiet", cwd=repo_path)        print(f"✅ Repository updated")    # Verify repository structure    if not (repo_path / "pyproject.toml").exists():        raise FileNotFoundError(            f"Repository structure invalid: pyproject.toml not found in {repo_path}. "            "The repository may be corrupted."        )    # Change working directory and update Python path    print(f"📁 Changing working directory to {repo_path}")    os.chdir(repo_path)    if str(repo_path) not in sys.path:        sys.path.insert(0, str(repo_path))    # Install dependencies (smart installation - only installs missing packages)    from core.notebook.setup import smart_install_dependencies    result = smart_install_dependencies(        repo_path=repo_path,        include_dev=True,        verbose=True,    )    # Fail loudly if critical packages failed to install    if result["failed"]:        print(f"⚠️  WARNING: {len(result['failed'])} packages failed to install:")        for pkg in result["failed"]:            print(f"  - {pkg}")        print("You may encounter import errors. Please check your internet connection.")    print("" + "=" * 70)    print("✅ Environment setup complete\! You can now proceed with the notebook.")    print("=" * 70)else:    print("💻 Running in local environment. Configuring...")    # Find the repository root    repo_path = find_repo_root(Path.cwd())    if repo_path is None:        raise FileNotFoundError(            "Could not find repository root (no pyproject.toml found). "            "Please ensure you are running this notebook from within the "            "ADH-LLM-Tutorials-2025 repository directory."        )    print(f"✅ Found repository root: {repo_path}")    # Change working directory and update Python path    print(f"📁 Changing working directory to {repo_path}")    os.chdir(repo_path)    if str(repo_path) not in sys.path:        sys.path.insert(0, str(repo_path))    print("" + "=" * 70)    print("✅ Local environment configured successfully\!")    print("=" * 70)    print("⚠️  Please ensure you have run: pip install -e .[dev]")    print("   (Required for local development)")

# 06 - Bias, Calibration, and Limitations

## Why "Accuracy" Isn't Enough: Evaluating Models for Clinical Deployment

In the previous notebook, we compared our models using AUROC and AUPRC—standard metrics that measure **discriminative performance**. But these metrics alone don't tell us if a model is **safe and trustworthy** for clinical use.

Consider two critical questions:

1. **Calibration**: Can we trust the model's predicted probabilities?  
   If the model predicts an 80% risk of sepsis, does that patient actually have an 80% chance of developing sepsis?

2. **Fairness**: Does the model work equally well for all patient subgroups?  
   Does it perform worse for elderly patients? For specific demographics?

These questions are not academic—they have **life-or-death consequences** in a clinical setting. This notebook teaches you to:

1. ✅ Assess **calibration** using reliability diagrams and Brier scores
2. ✅ Investigate **subgroup fairness** by comparing performance across demographic groups
3. ✅ Understand the **ethical implications** of deploying biased or miscalibrated models

By the end of this notebook, you'll understand why evaluating models for deployment requires more than just headline AUROC numbers.

In [ ]:
# Import required libraries
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import yaml

from core.config import TrainConfig, TransformerConfig
from core.data import build_patient_dataframe_for_subset, create_dataloaders
from core.data.physionet_sepsis import get_sepsis_data
from core.evaluation import (
    calculate_calibration_metrics,
    calculate_subgroup_metrics,
    get_predictions,
)
from core.models import TransformerModel
from core.notebook import ensure_project_root
from core.viz import plot_calibration_curve, plot_subgroup_performance

## Step 1: Load a Model and Generate Predictions

We'll use the **Transformer model** as the case study for this analysis.

In [ ]:
# Ensure execution happens from the project root
project_root = ensure_project_root()
models_dir = project_root / "models"
configs_dir = project_root / "configs"

# Define paths
transformer_model_path = models_dir / "transformer_best.pt"

# Determine device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load configuration
transformer_config = TransformerConfig(
    **yaml.safe_load((configs_dir / "transformer.yaml").read_text())["model"]
)
train_config = TrainConfig(
    **yaml.safe_load((configs_dir / "transformer.yaml").read_text())["training"]
)

# Instantiate and load model
transformer_model = TransformerModel(transformer_config)
transformer_model.load_state_dict(
    torch.load(transformer_model_path, map_location=device)
)
transformer_model.to(device)

print("✅ Transformer model loaded successfully!")

# Load data - we need the FULL DataFrame to access demographic columns
sepsis_df = get_sepsis_data()
train_loader, test_loader = create_dataloaders(train_config=train_config, df=sepsis_df)

print(f"\nTest set size: {len(test_loader.dataset)} patients")

In [ ]:
# Generate predictions
print("Generating predictions...")
preds = get_predictions(
    transformer_model,
    test_loader,
    device=device,
    return_tensors=True,
)

labels = preds.labels.numpy()
predictions = preds.probabilities.numpy()

print(f"✅ Generated {len(labels)} predictions")
print(f"   Positive class ratio: {labels.mean():.3f}")

## Step 2: Model Calibration - Can We Trust the Probabilities?

### What is Calibration?

A **well-calibrated** model produces predicted probabilities that match empirical event frequencies:
- If the model predicts 70% risk for 100 patients, approximately 70 of them should develop sepsis
- If probabilities are systematically too high, the model is **overconfident**
- If probabilities are systematically too low, the model is **underconfident**

### Why Calibration Matters in Healthcare

Imagine a clinician receives an alert: "Patient X has 85% risk of sepsis." If the model is poorly calibrated:
- The true risk might actually be 40% or 95%
- This leads to **inappropriate clinical actions**: unnecessary interventions or missed critical cases
- Trust in the system erodes

### Metrics We'll Use

1. **Reliability Diagram (Calibration Curve)**: Visual plot of predicted vs. observed probabilities
2. **Brier Score**: Mean squared error between predicted probabilities and true labels (lower is better)

In [ ]:
# Calculate calibration metrics
cal_metrics = calculate_calibration_metrics(labels, predictions)

print(f"Brier Score: {cal_metrics['brier_score']:.4f}")
print("\nInterpretation:")
print("  - Brier score ranges from 0 (perfect) to 1 (worst)")
print("  - A score < 0.10 is typically considered well-calibrated")
print("  - A score > 0.25 indicates serious calibration issues")

In [ ]:
# Plot calibration curve
fig = plot_calibration_curve(cal_metrics, model_name="Transformer")
plt.show()

### Analyzing the Calibration Curve

**What to look for:**

- **Points on the diagonal**: Perfect calibration—predicted probabilities match true event rates
- **Points above the diagonal**: Model is **underconfident** (predicts lower probabilities than reality)
- **Points below the diagonal**: Model is **overconfident** (predicts higher probabilities than reality)

**Common Issues:**

- Deep learning models often exhibit **overconfidence** due to overfitting
- Models trained on imbalanced data may show **poor calibration** in extreme probability ranges

**What can we do if calibration is poor?**

1. **Recalibration**: Apply post-hoc calibration methods (e.g., Platt scaling, isotonic regression)
2. **Ensemble methods**: Combine multiple models to improve calibration
3. **Regularization**: Add stronger regularization during training to reduce overconfidence
4. **Focal loss**: Use calibration-aware loss functions during training

## Step 3: Subgroup Fairness - Does the Model Work for Everyone?

### Why Subgroup Analysis is Essential

A model with **high overall AUROC** can still have **severe disparities** across patient subgroups:
- It might work well for young patients but fail for the elderly
- It might perform differently for patients in different hospital units
- It might have biases related to gender, race, or socioeconomic status

These disparities can:
- Lead to **unequal care** and worse outcomes for vulnerable populations
- Violate **ethical principles** of fairness in healthcare
- Create **legal and regulatory risks** for health systems

### Our Analysis: Performance by Age Group

We'll stratify patients by age and examine whether the model's discriminative performance (AUROC) is consistent across age groups.

In [ ]:
# Align patient-level metadata with the evaluation subset
test_subset = test_loader.dataset
test_df = build_patient_dataframe_for_subset(
    test_subset,
    sepsis_df,
    columns=["Age"],
)

if len(test_df) != len(labels):
    raise RuntimeError(
        "Mismatch between demographic rows and model predictions for the test set."
    )

# Create age groups
bins = [0, 40, 60, 80, 100]
labels_age = ["0-40", "41-60", "61-80", "81+"]
test_df["age_group"] = pd.cut(test_df["Age"], bins=bins, labels=labels_age, right=False)

print("Age group distribution in test set (aligned with predictions):")
print(test_df["age_group"].value_counts().sort_index())

In [ ]:
# Calculate subgroup performance
subgroup_auroc = calculate_subgroup_metrics(
    test_df, labels, predictions, subgroup_col="age_group"
)

print("\nAUROC by Age Group:")
for group, auroc in sorted(subgroup_auroc.items()):
    print(f"  {group}: {auroc:.4f}")

In [ ]:
# Visualize subgroup performance
fig = plot_subgroup_performance(subgroup_auroc)
plt.show()

### Interpreting Subgroup Performance Disparities

**Questions to ask:**

1. **How large are the disparities?**  
   - Small differences (< 0.05 AUROC) may be acceptable  
   - Large differences (> 0.10 AUROC) indicate serious fairness concerns

2. **Which groups are disadvantaged?**  
   - Is the model worse for older patients? (Common due to data imbalance)  
   - Is there a specific age range where performance drops?

3. **What are the clinical consequences?**  
   - If the model performs poorly for elderly patients (who have higher sepsis risk), this could lead to **missed diagnoses** in the most vulnerable population
   - Conversely, if the model is overconfident for younger patients, it could lead to **overtreatment**

### What to do if disparities are found?

1. **Data collection**: Ensure training data is representative of all patient subgroups
2. **Stratified sampling**: Oversample underrepresented groups during training
3. **Fairness-aware training**: Use loss functions that penalize subgroup disparities
4. **Group-specific thresholds**: Set different decision thresholds for different subgroups
5. **Report findings**: Transparently document performance disparities in deployment documentation

## Discussion: Beyond the Numbers

### Critical Reflection Questions

Consider the following scenarios:

1. **Scenario A**: Your model has an AUROC of 0.90 overall, but only 0.75 for patients aged 81+. What do you do?
   - Do you deploy the model as-is?
   - Do you exclude elderly patients from the model's predictions?
   - Do you collect more data and retrain?

2. **Scenario B**: Your model is well-calibrated on average (Brier score = 0.08) but overconfident at high predicted probabilities (> 0.8). What are the clinical implications?
   - Could this lead to unnecessary aggressive interventions?
   - Should you apply recalibration before deployment?

3. **Scenario C**: You discover that patients in Unit1 have higher AUROC than Unit2. What might explain this?
   - Data leakage (model learning hospital unit as a proxy for something else)?
   - Genuine clinical differences (Unit1 is specialized for a specific patient population)?
   - Differences in data quality or measurement practices?

### Ethical Takeaways

- **No model is perfect**: Every model will have limitations and biases
- **Transparency is essential**: Document and report calibration and fairness metrics
- **Context matters**: A model's suitability depends on the specific clinical context and patient population
- **Continuous monitoring**: Fairness and calibration should be monitored in production, not just during development

## Challenge: Investigate Other Subgroups

The PhysioNet sepsis dataset includes other demographic and clinical variables. Try investigating:

- **Gender**: Does the model perform equally for male and female patients?
- **Hospital Unit**: Are there differences in performance across ICU units?
- **ICULOS (ICU length of stay)**: Does performance degrade for patients who have been in the ICU longer?

**Example code to get you started:**

```python
# Investigate gender subgroups
gender_auroc = calculate_subgroup_metrics(
    test_df, labels, predictions, subgroup_col="Gender"
)
fig = plot_subgroup_performance(gender_auroc, metric_name="AUROC")
plt.show()
```

Think critically about what you find!

## Conclusion

In this notebook, we've gone beyond traditional performance metrics to evaluate the **trustworthiness and fairness** of our sepsis prediction model.

### What We Learned

1. ✅ **Calibration analysis** reveals whether we can trust the model's predicted probabilities
2. ✅ **Subgroup fairness analysis** identifies performance disparities that could lead to unequal care
3. ✅ **Critical evaluation** is essential before deploying models in high-stakes clinical settings

### The Bigger Picture

This notebook represents a critical mindset shift:
- Moving from **research experiments** to **responsible deployment**
- Recognizing that **high AUROC ≠ safe for clinical use**
- Embracing the **ethical responsibility** of building AI systems that affect human lives

### Next Steps in a Real Project

1. **Prospective validation**: Test the model on data from a different hospital or time period
2. **Clinical trial**: Conduct a randomized controlled trial to measure real-world impact
3. **Stakeholder engagement**: Present findings to clinicians, ethicists, and patients
4. **Continuous monitoring**: Track performance, calibration, and fairness in production

Congratulations on completing this critical module on bias, calibration, and limitations! You're now equipped to evaluate machine learning models with the rigor and responsibility required for healthcare applications. 🎉